# EGFR "Virtual Biopsy" — Multimodal (CT + Clinical) Classifier
### NSCLC-Radiogenomics · predict EGFR mutation status from a CT scan + clinical data

This notebook runs the whole pipeline end-to-end:
1. install dependencies
2. load the clinical labels + index the DICOM images
3. preprocess each CT (find the tumour, crop it, normalise) and cache it
4. build a **two-branch fusion model** (3D-CNN for the CT tumour + MLP for clinical data)
5. train with cross-validation and evaluate (AUROC / AUPRC / sensitivity / specificity)
6. compare **fusion vs clinical-only vs image-only** so we can see the multimodal model earns its keep

**What we predict:** EGFR mutation status — **Mutant (1) vs Wildtype (0)**. We drop patients labelled Unknown / Not collected.
**Why it matters:** EGFR-mutant tumours respond to targeted pills (TKIs), but confirming the mutation needs slow, costly DNA sequencing. A CT-based predictor can *triage* who to sequence first — useful especially where sequencing is scarce.

**All knobs you might tune live in the CONFIG cell.** Edit paths there first.


## 1. Install dependencies

`pydicom-seg` is **not** compatible with pydicom 3.x yet, so we pin `pydicom<3`. Run once.

In [1]:
# Run once. Safe to re-run.
import sys
# !{sys.executable} -m pip install -q "pydicom<3" pydicom-seg SimpleITK scipy scikit-learn matplotlib tqdm
# PyTorch: if not already installed, install the build matching your CUDA from pytorch.org. Example (CUDA 12.1):
# !{sys.executable} -m pip install torch --index-url https://download.pytorch.org/whl/cu121
print("done")

done


## 2. Imports

In [2]:
import os, glob, warnings, numpy as np, pandas as pd
from pathlib import Path
from collections import defaultdict

import pydicom
import pydicom_seg
import SimpleITK as sitk
from scipy import ndimage

import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             confusion_matrix, roc_curve)

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **k): return x

warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch", torch.__version__, "| Device:", DEVICE)

Torch 2.5.1+cu121 | Device: cuda


d:\git\PhDProjects\LungCancerGrading\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. CONFIG — every tunable knob lives here

Edit the **paths** to match your machine, then leave the rest at defaults for a first run.

In [3]:
# ---------- PATHS (EDIT THESE) ----------
DATA_ROOT    = Path("data/nsclc_radiogenomics")   # folder containing AMC-xxx / R01-xxx patient folders
CLINICAL_CSV = Path("data/NSCLCR01Radiogenomic_DATA_LABELS_2018-05-22_1500-shifted.csv")
CACHE_DIR    = Path("egfr_cache")                  # cached CT tumour cubes go here (created automatically)
OUTPUT_DIR   = Path("egfr_out")                    # models / results
CACHE_DIR.mkdir(exist_ok=True); OUTPUT_DIR.mkdir(exist_ok=True)

# ---------- TARGET ----------
TARGET_COL   = "EGFR mutation status"
POS_LABEL    = "Mutant"      # -> 1
NEG_LABEL    = "Wildtype"    # -> 0   (everything else, e.g. Unknown / Not collected, is dropped)

# ---------- IMAGE PREPROCESSING ----------
TARGET_SHAPE     = (32, 64, 64)   # (Depth, Height, Width) of the cropped tumour cube fed to the CNN
CROP_PAD_VOX     = 16             # voxels of context padding around the tumour bounding box
HU_MIN, HU_MAX   = -1000, 400     # Hounsfield-unit window (air .. soft tissue); values clipped to this range
REQUIRE_SEG      = True          # True = only keep patients with a usable tumour segmentation (cleaner, smaller N)
ALLOW_CENTER_CROP_FALLBACK = True # if no segmentation, fall back to a centred crop of the chest (weaker signal)

# ---------- TABULAR FEATURES ----------
INCLUDE_HISTOLOGY = False   # histology (adeno/squamous) is borderline-leaky for EGFR; keep False for the clean run

# ---------- MODEL ----------
IMG_FEAT_DIM   = 64    # size of the image branch's output vector
TAB_HIDDEN     = 64    # tabular MLP hidden width
FUSION_HIDDEN  = 64    # fusion head hidden width
DROPOUT        = 0.4   # high, because the dataset is small

# ---------- TRAINING ----------
N_FOLDS        = 5
EPOCHS         = 40
LR             = 3e-4
WEIGHT_DECAY   = 1e-4
BATCH_SIZE     = 8
USE_CLASS_WEIGHTS = True   # weight the loss to handle the ~25% positive imbalance
SEED           = 0

torch.manual_seed(SEED); np.random.seed(SEED)

## 4. Load clinical labels + build tabular features

We read the CSV, map EGFR to 0/1, drop unknown labels, and build a small vector of **safe** clinical features (known at scan time — no surgery/pathology fields, to avoid leakage).

In [4]:
clin = pd.read_csv(CLINICAL_CSV)
clin["Case ID"] = clin["Case ID"].astype(str).str.strip()

# map target -> 0/1, drop anything that isn't Mutant/Wildtype
clin["label"] = clin[TARGET_COL].map({POS_LABEL: 1, NEG_LABEL: 0})
clin = clin.dropna(subset=["label"]).copy()
clin["label"] = clin["label"].astype(int)
print("Patients with a usable EGFR label:", len(clin))
print(clin["label"].value_counts().rename({0: NEG_LABEL, 1: POS_LABEL}))

def build_tabular(df):
    f = pd.DataFrame(index=df.index)
    age = pd.to_numeric(df["Age at Histological Diagnosis"], errors="coerce")
    f["age"] = (age.fillna(age.median()) / 100.0)                       # deterministic scaling -> no fold leakage
    f["male"] = (df["Gender"].astype(str).str.strip().str.lower() == "male").astype(float)
    sm = df["Smoking status"].astype(str).str.strip()
    for v in ["Nonsmoker", "Former", "Current"]:
        f[f"smk_{v}"] = (sm == v).astype(float)
    for c in [c for c in df.columns if c.startswith("Tumor Location")]:
        key = c.split("choice=")[-1].rstrip(")").strip().replace(" ", "_")
        f[f"loc_{key}"] = (df[c].astype(str).str.strip().str.lower() == "checked").astype(float)
    gg = df["%GG"].astype(str).str.strip()
    f = pd.concat([f, pd.get_dummies(gg, prefix="gg").astype(float)], axis=1)
    if INCLUDE_HISTOLOGY:
        h = df["Histology "].astype(str).str.strip()
        f = pd.concat([f, pd.get_dummies(h, prefix="hist").astype(float)], axis=1)
    return f

tab_df = build_tabular(clin)
print("\nTabular feature columns (", tab_df.shape[1], "):")
print(list(tab_df.columns))

Patients with a usable EGFR label: 172
label
Wildtype    129
Mutant       43
Name: count, dtype: int64

Tabular feature columns ( 19 ):
['age', 'male', 'smk_Nonsmoker', 'smk_Former', 'smk_Current', 'loc_RUL', 'loc_RML', 'loc_RLL', 'loc_LUL', 'loc_LLL', 'loc_L_Lingula', 'loc_Unknown', 'gg_0%', 'gg_100%', 'gg_25 - 50%', 'gg_50 - 75%', 'gg_75 - < 100%', 'gg_>0 - 25%', 'gg_Not Assessed']


## 5. Index the DICOM images

Each patient folder holds several series with meaningless numeric names. We walk each folder and read just the **header** of one file per series to learn its **Modality** (CT / PT / SEG / RTSTRUCT). We keep the CT series with the most slices, and the segmentation object if present.

In [5]:
def scan_patient(pdir):
    info = {"ct_dir": None, "ct_n": 0, "seg_file": None, "seg_modality": None}
    for root, _, files in os.walk(pdir):
        dcms = [f for f in files if f.lower().endswith(".dcm")]
        if not dcms:
            continue
        try:
            ds = pydicom.dcmread(os.path.join(root, dcms[0]), stop_before_pixels=True, force=True)
        except Exception:
            continue
        mod = str(getattr(ds, "Modality", "")).upper()
        if mod == "CT":
            if len(dcms) > info["ct_n"]:
                info["ct_dir"], info["ct_n"] = root, len(dcms)
        elif mod in ("SEG", "RTSTRUCT"):
            info["seg_file"], info["seg_modality"] = os.path.join(root, dcms[0]), mod
    return info

records = []
for cid in tqdm(clin["Case ID"].tolist(), desc="indexing"):
    pdir = DATA_ROOT / cid
    if not pdir.exists():
        records.append({"Case ID": cid, "ct_dir": None, "ct_n": 0, "seg_file": None, "seg_modality": None})
        continue
    info = scan_patient(pdir); info["Case ID"] = cid
    records.append(info)

idx = pd.DataFrame(records)
cohort = clin.merge(idx, on="Case ID", how="left")
print("Have a CT series:        ", cohort["ct_dir"].notna().sum())
print("Have a SEG segmentation: ", (cohort["seg_modality"] == "SEG").sum())
print("Have an RTSTRUCT only:   ", (cohort["seg_modality"] == "RTSTRUCT").sum())
print("Have neither:            ", cohort["seg_modality"].isna().sum())

indexing: 100%|██████████| 172/172 [00:29<00:00,  5.81it/s]

Have a CT series:         172
Have a SEG segmentation:  117
Have an RTSTRUCT only:    0
Have neither:             55


## 6. Preprocess each CT → cropped tumour cube (cached to disk)

For every patient: load the CT volume (correctly spaced, in Hounsfield Units), find the tumour from the segmentation, crop a padded box around it, resize to a fixed cube, window + normalise, and save as a small `.npy`. This is the one-time heavy step; training afterwards is light.

If a patient has no usable segmentation and `ALLOW_CENTER_CROP_FALLBACK=True`, we fall back to a centred chest crop (weaker — flagged per patient).

In [6]:
def load_ct(ct_dir):
    r = sitk.ImageSeriesReader()
    files = r.GetGDCMSeriesFileNames(str(ct_dir))
    r.SetFileNames(files)
    return r.Execute()                       # SimpleITK image, intensities in HU, [Z,Y,X] when arrayed

def seg_mask_on_ct(seg_file, ct_img):
    # returns a boolean mask array aligned to the CT, or None on failure
    dcm = pydicom.dcmread(seg_file)
    result = pydicom_seg.SegmentReader().read(dcm)
    mask = None
    for num in result.available_segments:
        seg_img = result.segment_image(num)
        res = sitk.Resample(seg_img, ct_img, sitk.Transform(), sitk.sitkNearestNeighbor, 0, sitk.sitkUInt8)
        a = sitk.GetArrayFromImage(res).astype(bool)
        mask = a if mask is None else (mask | a)
    return mask

def crop_to_cube(ct_arr, mask):
    # crop around the tumour bbox (mask) or centre of the volume (mask is None), then resize to TARGET_SHAPE
    if mask is not None and mask.any():
        zz, yy, xx = np.where(mask)
        z0, z1 = zz.min(), zz.max(); y0, y1 = yy.min(), yy.max(); x0, x1 = xx.min(), xx.max()
        p = CROP_PAD_VOX
        z0, z1 = max(0, z0 - p), min(ct_arr.shape[0], z1 + p + 1)
        y0, y1 = max(0, y0 - p), min(ct_arr.shape[1], y1 + p + 1)
        x0, x1 = max(0, x0 - p), min(ct_arr.shape[2], x1 + p + 1)
    else:
        Z, Y, X = ct_arr.shape           # centred fallback crop (central 60% in-plane, central 50% in Z)
        z0, z1 = int(Z*0.25), int(Z*0.75); y0, y1 = int(Y*0.2), int(Y*0.8); x0, x1 = int(X*0.2), int(X*0.8)
    crop = ct_arr[z0:z1, y0:y1, x0:x1].astype(np.float32)
    factors = [t / max(1, s) for t, s in zip(TARGET_SHAPE, crop.shape)]
    crop = ndimage.zoom(crop, factors, order=1)
    crop = np.clip(crop, HU_MIN, HU_MAX)
    crop = (crop - HU_MIN) / (HU_MAX - HU_MIN)    # -> [0,1]
    return crop.astype(np.float32)

status = []
for _, row in tqdm(cohort.iterrows(), total=len(cohort), desc="preprocess"):
    cid = row["Case ID"]; out = CACHE_DIR / f"{cid}.npy"
    if out.exists():
        status.append((cid, "cached", True)); continue
    if pd.isna(row["ct_dir"]):
        status.append((cid, "no_ct", False)); continue
    try:
        ct = load_ct(row["ct_dir"]); ct_arr = sitk.GetArrayFromImage(ct)
        mask, seg_used = None, False
        if row["seg_modality"] == "SEG":
            try:
                mask = seg_mask_on_ct(row["seg_file"], ct); seg_used = mask is not None and mask.any()
            except Exception:
                mask = None
        if (mask is None or not mask.any()):
            if not ALLOW_CENTER_CROP_FALLBACK or REQUIRE_SEG:
                status.append((cid, "no_seg_skipped", False)); continue
        cube = crop_to_cube(ct_arr, mask if (mask is not None and mask.any()) else None)
        np.save(out, cube)
        status.append((cid, "ok_seg" if seg_used else "ok_centercrop", seg_used))
    except Exception as e:
        status.append((cid, f"error:{type(e).__name__}", False))

st = pd.DataFrame(status, columns=["Case ID", "status", "seg_used"])
print(st["status"].value_counts())
cohort = cohort.merge(st, on="Case ID", how="left")
cohort = cohort[cohort["status"].isin(["ok_seg", "ok_centercrop", "cached"])].reset_index(drop=True)
print("\nFinal usable cohort:", len(cohort), "| label balance:")
print(cohort["label"].value_counts().rename({0: NEG_LABEL, 1: POS_LABEL}))

preprocess: 100%|██████████| 172/172 [21:45<00:00,  7.59s/it]

status
ok_seg            111
no_seg_skipped     61
Name: count, dtype: int64

Final usable cohort: 111 | label balance:
label
Wildtype    90
Mutant      21
Name: count, dtype: int64


## 7. PyTorch Dataset

Loads each patient's cached cube `[1,D,H,W]` and clinical vector. The tabular matrix is aligned to the cohort rows.

In [7]:
tab_mat = build_tabular(cohort).values.astype(np.float32)   # rebuild aligned to final cohort order
N_TAB = tab_mat.shape[1]
labels = cohort["label"].values.astype(np.float32)
print("Tabular feature dim:", N_TAB)

class EGFRDataset(Dataset):
    def __init__(self, indices):
        self.indices = list(indices)
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, k):
        i = self.indices[k]
        cube = np.load(CACHE_DIR / f"{cohort.iloc[i]['Case ID']}.npy")  # [D,H,W]
        img = torch.from_numpy(cube).float().unsqueeze(0)               # [1,D,H,W]
        tab = torch.from_numpy(tab_mat[i]).float()
        y   = torch.tensor([labels[i]], dtype=torch.float32)
        return img, tab, y

Tabular feature dim: 18


## 8. The model — two-branch fusion (3D-CNN + tabular MLP)

The image branch is a small 3D CNN that turns the tumour cube into a vector; the tabular branch is an MLP. We concatenate the two vectors and a small head outputs one logit (EGFR+ score). The `use_image` / `use_tabular` switches let us run the same code for the ablation (image-only, clinical-only, fusion).

In [8]:
class Image3DEncoder(nn.Module):
    def __init__(self, out_dim=IMG_FEAT_DIM, p=DROPOUT):
        super().__init__()
        def blk(ci, co):
            return nn.Sequential(nn.Conv3d(ci, co, 3, padding=1), nn.BatchNorm3d(co),
                                 nn.ReLU(inplace=True), nn.MaxPool3d(2))
        self.net = nn.Sequential(blk(1, 16), blk(16, 32), blk(32, 64),
                                 nn.AdaptiveAvgPool3d(1), nn.Flatten())
        self.proj = nn.Sequential(nn.Dropout(p), nn.Linear(64, out_dim), nn.ReLU(inplace=True))
    def forward(self, x):
        return self.proj(self.net(x))               # [B, out_dim]

class TabularEncoder(nn.Module):
    def __init__(self, n_in, hidden=TAB_HIDDEN, out_dim=32, p=DROPOUT):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_in, hidden), nn.ReLU(inplace=True), nn.Dropout(p),
                                 nn.Linear(hidden, out_dim), nn.ReLU(inplace=True))
    def forward(self, x):
        return self.net(x)                          # [B, out_dim]

class FusionNet(nn.Module):
    def __init__(self, n_tab, use_image=True, use_tabular=True):
        super().__init__()
        self.use_image, self.use_tabular = use_image, use_tabular
        dim = 0
        if use_image:   self.img = Image3DEncoder();           dim += IMG_FEAT_DIM
        if use_tabular: self.tab = TabularEncoder(n_tab);      dim += 32
        self.head = nn.Sequential(nn.Linear(dim, FUSION_HIDDEN), nn.ReLU(inplace=True),
                                  nn.Dropout(DROPOUT), nn.Linear(FUSION_HIDDEN, 1))
    def forward(self, img, tab):
        feats = []
        if self.use_image:   feats.append(self.img(img))
        if self.use_tabular: feats.append(self.tab(tab))
        return self.head(torch.cat(feats, dim=1))   # [B,1] logit

print("FusionNet params:",
      sum(p.numel() for p in FusionNet(N_TAB).parameters()))

FusionNet params: 83617


## 9. Train + predict helpers

In [9]:
from torch.cuda.amp import autocast, GradScaler

@torch.no_grad()
def predict(model, loader):
    model.eval(); P, Y = [], []
    for img, tab, y in loader:
        img, tab = img.to(DEVICE), tab.to(DEVICE)
        with autocast(enabled=(DEVICE == "cuda")):
            logit = model(img, tab)
        P.append(torch.sigmoid(logit).float().cpu().numpy().ravel())
        Y.append(y.numpy().ravel())
    return np.concatenate(P), np.concatenate(Y)

def train_one(train_idx, val_idx, use_image=True, use_tabular=True, verbose=False):
    tr = DataLoader(EGFRDataset(train_idx), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    va = DataLoader(EGFRDataset(val_idx),   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    model = FusionNet(N_TAB, use_image, use_tabular).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = GradScaler(enabled=(DEVICE == "cuda"))
    if USE_CLASS_WEIGHTS:
        n_pos = labels[train_idx].sum(); n_neg = len(train_idx) - n_pos
        pos_weight = torch.tensor([n_neg / max(1.0, n_pos)], device=DEVICE)
    else:
        pos_weight = None
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_auc, best_state = -1.0, None
    for ep in range(EPOCHS):
        model.train()
        for img, tab, y in tr:
            img, tab, y = img.to(DEVICE), tab.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            with autocast(enabled=(DEVICE == "cuda")):
                loss = crit(model(img, tab), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        p, yt = predict(model, va)
        auc = roc_auc_score(yt, p) if len(np.unique(yt)) > 1 else 0.5
        if auc > best_auc:
            best_auc = auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if verbose:
            print(f"  epoch {ep:02d}  val AUROC {auc:.3f}")
    model.load_state_dict(best_state)
    return model

## 10. Cross-validation

5-fold, stratified jointly on **label and site** (so EGFR-positives and the two hospitals stay balanced across folds). Every patient is predicted once, in the fold where it's held out (out-of-fold), giving honest overall metrics.

In [10]:
def run_cv(use_image=True, use_tabular=True, tag="fusion"):
    strat = (cohort["label"].astype(str) + "_" + cohort["Patient affiliation"].astype(str)).values
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    oof = np.zeros(len(cohort)); fold_auc = []
    for f, (tr, va) in enumerate(skf.split(np.arange(len(cohort)), strat)):
        model = train_one(tr, va, use_image, use_tabular)
        p, yt = predict(model, DataLoader(EGFRDataset(va), batch_size=BATCH_SIZE))
        oof[va] = p
        a = roc_auc_score(yt, p) if len(np.unique(yt)) > 1 else 0.5
        fold_auc.append(a)
        print(f"[{tag}] fold {f}  AUROC {a:.3f}")
    print(f"[{tag}] mean fold AUROC {np.mean(fold_auc):.3f} +/- {np.std(fold_auc):.3f}")
    return oof

oof_fusion = run_cv(use_image=True, use_tabular=True, tag="fusion")

[fusion] fold 0  AUROC 0.794
[fusion] fold 1  AUROC 0.625
[fusion] fold 2  AUROC 0.569
[fusion] fold 3  AUROC 0.611
[fusion] fold 4  AUROC 0.806
[fusion] mean fold AUROC 0.681 +/- 0.099


## 11. Evaluation

Overall out-of-fold metrics: **AUROC** (ranking quality), **AUPRC** (better than AUROC under class imbalance), and **sensitivity / specificity** at the threshold that maximises Youden's J (sens+spec-1). We also print the confusion matrix.

In [11]:
def evaluate(oof, name="model"):
    y = labels.astype(int)
    auroc = roc_auc_score(y, oof)
    auprc = average_precision_score(y, oof)
    fpr, tpr, thr = roc_curve(y, oof)
    j = np.argmax(tpr - fpr); t = thr[j]
    pred = (oof >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    sens = tp / (tp + fn); spec = tn / (tn + fp)
    print(f"=== {name} ===")
    print(f"  AUROC {auroc:.3f} | AUPRC {auprc:.3f}  (positive prevalence {y.mean():.2f})")
    print(f"  @threshold {t:.2f}:  sensitivity {sens:.2f}  specificity {spec:.2f}")
    print(f"  confusion [[TN {tn}, FP {fp}], [FN {fn}, TP {tp}]]")
    return dict(auroc=auroc, auprc=auprc, sens=sens, spec=spec)

_ = evaluate(oof_fusion, "FUSION (CT + clinical)")

=== FUSION (CT + clinical) ===
  AUROC 0.676 | AUPRC 0.322  (positive prevalence 0.19)
  @threshold 0.44:  sensitivity 0.81  specificity 0.50
  confusion [[TN 45, FP 45], [FN 4, TP 17]]


## 12. Ablation — does multimodal actually help?

The key scientific question. We compare three models on the same folds:
- **clinical-only** (a fast logistic-regression baseline + the deep clinical branch),
- **image-only** (CT tumour cube only),
- **fusion** (both).

If fusion doesn't beat the better single modality, the extra complexity isn't justified — and that's an honest, publishable finding in itself.

In [12]:
# fast sklearn clinical-only baseline (sanity bar)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
oof_lr = cross_val_predict(lr, tab_mat, labels.astype(int), cv=N_FOLDS, method="predict_proba")[:, 1]
print("clinical-only (logreg) AUROC:", round(roc_auc_score(labels.astype(int), oof_lr), 3))

# deep image-only and clinical-only (same architecture, switches flipped)
oof_img  = run_cv(use_image=True,  use_tabular=False, tag="image-only")
oof_clin = run_cv(use_image=False, use_tabular=True,  tag="clinical-only")

print()
for nm, o in [("clinical-only (logreg)", oof_lr),
              ("clinical-only (deep)",   oof_clin),
              ("image-only (deep)",      oof_img),
              ("FUSION",                 oof_fusion)]:
    print(f"{nm:24s}  AUROC {roc_auc_score(labels.astype(int), o):.3f}  AUPRC {average_precision_score(labels.astype(int), o):.3f}")

clinical-only (logreg) AUROC: 0.641
[image-only] fold 0  AUROC 0.822
[image-only] fold 1  AUROC 0.653
[image-only] fold 2  AUROC 0.694
[image-only] fold 3  AUROC 0.639
[image-only] fold 4  AUROC 0.889
[image-only] mean fold AUROC 0.739 +/- 0.099
[clinical-only] fold 0  AUROC 0.800
[clinical-only] fold 1  AUROC 0.590
[clinical-only] fold 2  AUROC 0.819
[clinical-only] fold 3  AUROC 0.431
[clinical-only] fold 4  AUROC 0.806
[clinical-only] mean fold AUROC 0.689 +/- 0.155

clinical-only (logreg)    AUROC 0.641  AUPRC 0.331
clinical-only (deep)      AUROC 0.562  AUPRC 0.224
image-only (deep)         AUROC 0.662  AUPRC 0.341
FUSION                    AUROC 0.676  AUPRC 0.322


## 13. Save the fusion model + single-patient inference

Retrain on all data and save, then show how to score one patient.

In [13]:
# retrain fusion on ALL patients and save
all_idx = np.arange(len(cohort))
final_model = train_one(all_idx, all_idx[:max(2, len(all_idx)//5)], use_image=True, use_tabular=True)
torch.save(final_model.state_dict(), OUTPUT_DIR / "egfr_fusion.pt")
print("saved", OUTPUT_DIR / "egfr_fusion.pt")

@torch.no_grad()
def predict_patient(case_id):
    i = cohort.index[cohort["Case ID"] == case_id][0]
    img = torch.from_numpy(np.load(CACHE_DIR / f"{case_id}.npy")).float().unsqueeze(0).unsqueeze(0).to(DEVICE)
    tab = torch.from_numpy(tab_mat[i]).float().unsqueeze(0).to(DEVICE)
    final_model.eval()
    prob = torch.sigmoid(final_model(img, tab)).item()
    return {"Case ID": case_id, "P(EGFR-mutant)": round(prob, 3),
            "true_label": POS_LABEL if labels[i] == 1 else NEG_LABEL}

print(predict_patient(cohort.iloc[0]["Case ID"]))

saved egfr_out\egfr_fusion.pt
{'Case ID': 'R01-003', 'P(EGFR-mutant)': 0.72, 'true_label': 'Mutant'}


## 14. Notes — what to tune, and what to improve next

**Easy knobs (in CONFIG):** `TARGET_SHAPE` (bigger cube = more detail, more memory), `CROP_PAD_VOX`, `EPOCHS`, `LR`, `DROPOUT`, `BATCH_SIZE`, `INCLUDE_HISTOLOGY`, `REQUIRE_SEG`.

**Things this v1 deliberately keeps simple (good next steps):**
- **Segmentation coverage.** Check the `status` counts in Step 6. Patients on the `ok_centercrop` fallback have a much weaker (non-tumour-focused) image — for the clean scientific run set `REQUIRE_SEG=True` and report on that subset. RTSTRUCT-only patients are currently skipped (would need the `rt-utils` library to convert contours to a mask).
- **In-fold standardisation.** Age is scaled deterministically here; proper practice fits the scaler on each fold's training data only.
- **Semantic-feature baseline.** The AIM XML files hold the 89 radiologist features that reached ~0.89 AUROC in the original paper — a strong baseline worth parsing.
- **PET branch / RNA-seq branch.** Add as third/fourth modalities for the research variant.
- **Augmentation + uncertainty.** 3D flips/rotations help small data; MC-dropout gives a confidence estimate for a "refer to sequencing" flag.
- **External validation.** Validate on a separate EGFR-CT cohort to prove generalisation.

**Reality check:** with ~150–170 patients and ~25% positives, expect AUROC with wide error bars. Judge success by **fusion vs clinical-only** (does the image add anything?) and by AUPRC, not by a single headline number.
